# AdventureWorks Sales – 01 Queries

Notebook for pulling core sales tables from SQL Server into pandas DataFrames.


In [19]:
# Core EDA setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Pandas display options
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

print("EDA environment ready.")

EDA environment ready.


In [20]:
import sqlalchemy as sa
import urllib

# Connection details
server = r"MYDELL23\SQLEXPRESS01"
database = "AdventureWorks"

conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    f"Server={server};"
    f"Database={database};"
    "Trusted_Connection=yes;"
)

params = urllib.parse.quote_plus(conn_str)
engine = sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query against AdventureWorks and return a DataFrame."""
    return pd.read_sql(sql, engine)


In [21]:
sql_monthly_top10 = """
SELECT TOP 10
    CONVERT(date, DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1)) AS MonthStart,
    SUM(sod.LineTotal) AS TotalSales
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail AS sod
    ON soh.SalesOrderID = sod.SalesOrderID
GROUP BY
    YEAR(soh.OrderDate),
    MONTH(soh.OrderDate)
ORDER BY TotalSales DESC;
"""

df_monthly_top10 = run_query(sql_monthly_top10)
df_monthly_top10.head()


,MonthStart,TotalSales
0,2014-03-01,"7,217,531.09"
1,2014-05-01,"5,366,674.97"
2,2013-06-01,"5,081,069.13"
3,2013-07-01,"4,896,353.74"
4,2013-10-01,"4,795,813.29"


In [22]:
sql_top_products = """
SELECT TOP 10
    p.Name AS ProductName,
    SUM(sod.LineTotal) AS TotalSales,
    SUM(sod.OrderQty) AS TotalQuantity,
    COUNT(DISTINCT sod.SalesOrderID) AS OrderCount
FROM Sales.SalesOrderDetail AS sod
JOIN Production.Product AS p
    ON sod.ProductID = p.ProductID
GROUP BY
    p.Name
ORDER BY TotalSales DESC;
"""

df_top_products = run_query(sql_top_products)
df_top_products.head()


,ProductName,TotalSales,TotalQuantity,OrderCount
0,"Mountain-200 Black, 38","4,400,592.80",2977,1252
1,"Mountain-200 Black, 42","4,009,494.76",2664,1177
2,"Mountain-200 Silver, 38","3,693,678.03",2394,1094
3,"Mountain-200 Silver, 42","3,438,478.86",2234,1040
4,"Mountain-200 Silver, 46","3,434,256.94",2216,1054


In [23]:
sql_territory_sales = """
SELECT TOP 10
    st.Name AS TerritoryName,
    SUM(soh.SubTotal) AS TotalSales,
    COUNT(DISTINCT soh.SalesOrderID) AS OrderCount,
    AVG(soh.SubTotal) AS AvgOrderValue
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesTerritory AS st
    ON soh.TerritoryID = st.TerritoryID
GROUP BY
    st.Name
ORDER BY TotalSales DESC;
"""

df_territory_sales = run_query(sql_territory_sales)
df_territory_sales.head()


,TerritoryName,TotalSales,OrderCount,AvgOrderValue
0,Southwest,"24,184,609.60",6224,"3,885.70"
1,Canada,"16,355,770.46",4067,"4,021.58"
2,Northwest,"16,084,942.55",4594,"3,501.29"
3,Australia,"10,655,335.96",6843,"1,557.11"
4,Central,"7,909,009.01",385,"20,542.88"


In [24]:
sql_sales_lines = """
SELECT
    sod.SalesOrderID,
    sod.SalesOrderDetailID,
    soh.OrderDate,
    sod.OrderQty,
    sod.UnitPrice,
    sod.LineTotal,
    soh.Status,
    p.ProductID,
    p.Name AS ProductName,
    ps.Name AS ProductSubcategoryName,
    pc.Name AS ProductCategoryName
FROM Sales.SalesOrderDetail AS sod
JOIN Sales.SalesOrderHeader AS soh
    ON sod.SalesOrderID = soh.SalesOrderID
JOIN Production.Product AS p
    ON sod.ProductID = p.ProductID
LEFT JOIN Production.ProductSubcategory AS ps
    ON p.ProductSubcategoryID = ps.ProductSubcategoryID
LEFT JOIN Production.ProductCategory AS pc
    ON ps.ProductCategoryID = pc.ProductCategoryID;
"""

df_sales_lines = run_query(sql_sales_lines)
df_sales_lines.head()


,SalesOrderID,SalesOrderDetailID,OrderDate,OrderQty,UnitPrice,LineTotal,Status,ProductID,ProductName,ProductSubcategoryName,ProductCategoryName
0,43659,1,2011-05-31,1,"2,024.99","2,024.99",5,776,"Mountain-100 Black, 42",Mountain Bikes,Bikes
1,43659,2,2011-05-31,3,"2,024.99","6,074.98",5,777,"Mountain-100 Black, 44",Mountain Bikes,Bikes
2,43659,3,2011-05-31,1,"2,024.99","2,024.99",5,778,"Mountain-100 Black, 48",Mountain Bikes,Bikes
3,43659,4,2011-05-31,1,"2,039.99","2,039.99",5,771,"Mountain-100 Silver, 38",Mountain Bikes,Bikes
4,43659,5,2011-05-31,1,"2,039.99","2,039.99",5,772,"Mountain-100 Silver, 42",Mountain Bikes,Bikes


In [25]:
# Apply cleaning function and save with ALL columns
from sys import path
path.append('../notebooks')  # Adjust if needed

# Simple cleaning - just filter out cancelled orders and convert date
clean_df = df_sales_lines.copy()
clean_df['OrderDate'] = pd.to_datetime(clean_df['OrderDate'])
clean_df = clean_df[clean_df['Status'] != 6]  # Exclude cancelled (Status=6)

# Create directory if needed
import os
os.makedirs("../data/clean", exist_ok=True)

# Save ALL columns including ProductCategoryName and ProductSubcategoryName
clean_df.to_parquet(
    "../data/clean/clean_sales_lines.parquet",
    engine="fastparquet",
    index=False
)

print(f"Saved clean_sales_lines.parquet with {len(clean_df)} rows and columns: {list(clean_df.columns)}")

Saved clean_sales_lines.parquet with 121317 rows and columns: ['SalesOrderID', 'SalesOrderDetailID', 'OrderDate', 'OrderQty', 'UnitPrice', 'LineTotal', 'Status', 'ProductID', 'ProductName', 'ProductSubcategoryName', 'ProductCategoryName']


## Data overview

Quick structural checks for the three core sales DataFrames used in this EDA:
- `df_monthly_top10`: top 10 months by total sales.
- `df_top_products`: top 10 products by sales, quantity, and order count.
- `df_territory_sales`: top 10 territories by total sales and average order value.

In [26]:
print("df_monthly_top10:", df_monthly_top10.shape)
display(df_monthly_top10.head())

print("\ndf_top_products:", df_top_products.shape)
display(df_top_products.head())

print("\ndf_territory_sales:", df_territory_sales.shape)
display(df_territory_sales.head())

df_monthly_top10: (10, 2)


,MonthStart,TotalSales
0,2014-03-01,"7,217,531.09"
1,2014-05-01,"5,366,674.97"
2,2013-06-01,"5,081,069.13"
3,2013-07-01,"4,896,353.74"
4,2013-10-01,"4,795,813.29"



df_top_products: (10, 4)


,ProductName,TotalSales,TotalQuantity,OrderCount
0,"Mountain-200 Black, 38","4,400,592.80",2977,1252
1,"Mountain-200 Black, 42","4,009,494.76",2664,1177
2,"Mountain-200 Silver, 38","3,693,678.03",2394,1094
3,"Mountain-200 Silver, 42","3,438,478.86",2234,1040
4,"Mountain-200 Silver, 46","3,434,256.94",2216,1054



df_territory_sales: (10, 4)


,TerritoryName,TotalSales,OrderCount,AvgOrderValue
0,Southwest,"24,184,609.60",6224,"3,885.70"
1,Canada,"16,355,770.46",4067,"4,021.58"
2,Northwest,"16,084,942.55",4594,"3,501.29"
3,Australia,"10,655,335.96",6843,"1,557.11"
4,Central,"7,909,009.01",385,"20,542.88"


In [27]:
def summarize_df(df, name):
    print(f"=== {name} ===")
    display(df.info())
    display(df.describe(include="all"))
    print("\nMissing values:")
    display(df.isna().sum())
    print("\n" + "-" * 60 + "\n")

summarize_df(df_monthly_top10, "df_monthly_top10")
summarize_df(df_top_products, "df_top_products")
summarize_df(df_territory_sales, "df_territory_sales")

=== df_monthly_top10 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MonthStart  10 non-null     object 
 1   TotalSales  10 non-null     float64
dtypes: float64(1), object(1)
memory usage: 292.0+ bytes


None

,MonthStart,TotalSales
count,10,10.00
unique,10,NaN
top,2014-03-01,NaN
freq,1,NaN
mean,NaN,"4,894,377.17"
std,NaN,"916,325.82"
min,NaN,"4,075,486.63"
25%,NaN,"4,350,590.64"
50%,NaN,"4,692,287.55"
75%,NaN,"5,034,890.28"



Missing values:


MonthStart    0
TotalSales    0
dtype: int64

...

None

,TerritoryName,TotalSales,OrderCount,AvgOrderValue
count,10,10.00,10.00,10.00
unique,10,NaN,NaN,NaN
top,Southwest,NaN,NaN,NaN
freq,1,NaN,NaN,NaN
mean,NaN,"10,984,638.14","3,146.50","7,640.68"
std,NaN,"6,022,438.94","2,335.61","7,833.25"
min,NaN,"4,915,407.60",352.00,"1,557.11"
25%,NaN,"7,356,346.99","1,020.25","2,465.69"
50%,NaN,"7,894,332.04","2,945.50","3,693.50"
75%,NaN,"14,727,540.90","4,462.25","13,165.36"



Missing values:


TerritoryName    0
TotalSales       0
OrderCount       0
AvgOrderValue    0
dtype: int64


------------------------------------------------------------

